# SLAM web app launcher

This notebook starts the same Dash SLAM web app implemented in `../scripts/slam_web_app.py`. The notebook itself does not render the point cloud map; open the web app URL after starting it.

Load the SLAM web app module and choose the DDS/network settings.

In [ ]:
import os
import socket
import sys
import threading
import time
from pathlib import Path

NOTEBOOK_DIR = Path.cwd().resolve()
MODULES_DIR = NOTEBOOK_DIR.parent
SCRIPTS_DIR = MODULES_DIR / "scripts"
for path in (str(MODULES_DIR), str(SCRIPTS_DIR)):
    if path not in sys.path:
        sys.path.insert(0, path)

IFACE = os.environ.get("G1_IFACE", "eth0")
DOMAIN_ID = int(os.environ.get("G1_DOMAIN_ID", "0"))
HOST = os.environ.get("G1_SLAM_WEB_HOST", "0.0.0.0")
PORT = int(os.environ.get("G1_SLAM_WEB_PORT", "8060"))
MAP_PATH = os.environ.get("G1_SLAM_MAP_PATH", "/home/unitree/test.pcd")

from slam_web_app import DEFAULT_TOPICS, SlamWebState, create_dash_app

print(f"Configured for iface={IFACE!r}, domain_id={DOMAIN_ID}, host={HOST!r}, port={PORT}.")

Create the web app state and start Dash in a background thread. If the default port is busy, change `PORT` above before running this cell.

In [ ]:
def port_is_available(host, port):
    bind_host = "" if host in ("0.0.0.0", "::") else host
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        sock.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
        try:
            sock.bind((bind_host, int(port)))
        except OSError:
            return False
    return True


if not port_is_available(HOST, PORT):
    raise RuntimeError(f"Port {PORT} is already in use. Set G1_SLAM_WEB_PORT or edit PORT above.")

slam_state = SlamWebState(IFACE, DOMAIN_ID, dict(DEFAULT_TOPICS), MAP_PATH)
slam_app = create_dash_app(slam_state)


def run_slam_web_app():
    slam_app.run(host=HOST, port=PORT, debug=False, use_reloader=False)


slam_web_thread = threading.Thread(target=run_slam_web_app, name="slam-web-app", daemon=True)
slam_web_thread.start()
time.sleep(1.0)

open_host = "127.0.0.1" if HOST == "0.0.0.0" else HOST
print(f"SLAM web app running at http://{open_host}:{PORT}")
print(f"map_path={MAP_PATH}")

Optional status check.

In [ ]:
print("thread_alive=", slam_web_thread.is_alive())
print("last_action=", slam_state.last_action)